# Handling large documents

In [1]:
import os, requests, io
from urllib.parse import urlparse
from PyPDF2 import PdfReader, PdfWriter 
from pathlib import Path

In [ ]:
def get_pdf_reader(url):
    base_filename = "output"
    response = requests.get(url, stream=True, timeout=30)
    response.raise_for_status()

    parsed_url = urlparse(url)
    path_part = os.path.basename(parsed_url.path)
    if path_part and '.' in path_part:
        base_filename = os.path.splitext(path_part)[0]
    
    pdf_content = io.BytesIO(response.content)
    reader = PdfReader(pdf_content)
    total_pages = len(reader.pages)
    return reader, base_filename, total_pages

In [ ]:
def split_pdf(url, output_dir, pages_per_chunk):
    reader, base_filename, total_pages = get_pdf_reader(url)

    if reader is None:
        print("Failed to get PDF reader. Aborting split...")
        return
    

    try:
        output_dir = Path(os.path.join(Path.cwd(), output_dir))
        output_dir.mkdir(exist_ok=True)

        num_chunks = (total_pages + pages_per_chunk - 1) // pages_per_chunk
        print(f"Splitting into {num_chunks} chunks of maximum {pages_per_chunk} pages each.")

        for i in range(num_chunks):
            writer = PdfWriter()
            start_page = i * pages_per_chunk
            end_page = min(start_page + pages_per_chunk, total_pages)

            print(f"Processing chunk {i+1}/{num_chunks} (pages {start_page-1}-{end_page})...")
            
            for page_num in range(start_page, end_page):
                writer.add_page(reader.pages[page_num])

            output_filename = os.path.join(output_dir, f"{base_filename}_chunk_{i+1}.pdf")

            with open(output_filename, 'wb') as f:
                writer.write(f)
            print(f"Chunk {i+1} saved as '{output_filename}'")

        print("\nPDF splitting completed successfully!")

    except Exception as e:
        print(f"An error occurred during PDF splitting: {str(e)}")

In [4]:
split_pdf("https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf", 
          output_dir='pdf_docs', 
          pages_per_chunk=50)

Splitting into 8 chunks of maximum 50 pages each.
Processing chunk 1/8 (pages -1-50)...
Chunk 1 saved as 'c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\Hands-On RAG for Production\Ch2\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_1.pdf'
Processing chunk 2/8 (pages 49-100)...
Chunk 2 saved as 'c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\Hands-On RAG for Production\Ch2\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_2.pdf'
Processing chunk 3/8 (pages 99-150)...
Chunk 3 saved as 'c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\Hands-On RAG for Production\Ch2\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_3.pdf'
Processing chunk 4/8 (pages 149-200)...
Chunk 4 saved as 'c:\Users\dorot\OneDrive - Politecnico di Torino\Desktop\Progetti\RAG_Books\Hands-On RAG for Production\Ch2\pdf_docs\SuttonBartoIPRLBook2ndEd_chunk_4.pdf'
Processing chunk 5/8 (pages 199-250)...
Chunk 5 saved as 'c:\Users\dorot\OneDrive - Politecnico di Torino\

In [5]:
url = "https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf"
local_file = "sutter_barto.pdf"
with requests.get(url, stream=True, timeout=30) as res:
    res.raise_for_status()
    with open(local_file, 'wb') as f:
        for chunk in res.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

In [7]:
!pip install docling

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   ---------------------------------------  10.2/10.4 MB 45.5 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 40.4 MB/s  0:00:00
   ---------------------------------------- 0.0/3.7 MB ? eta -:--:--
   ---------------------------------------- 3.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.3.0 requires dill<0.4.1,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.3.0 requires fsspec[http]<=2025.9.0,>=2023.1.0, but you have fsspec 2026.2.0 which is incompatible.
datasets 4.3.0 requires multiprocess<0.70.17, but you have multiprocess 0.70.19 which is incompatible.
llama-index-core 0.10.68.post1 requires numpy<2.0.0, but you have numpy 2.2.6 which is incompatible.
llama-index-vector-stores-deeplake 0.5.0 requires llama-index-core<0.15,>=0.13.0, but you have llama-index-core 0.10.68.post1 which is incompatible.
typer-slim 0.24.0 requires typer>=0.24.0, but you have typer 0.21.2 which is incompatible.


In [10]:
import os
os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'

In [11]:
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(generate_picture_images=True)
res = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options),
    }
).convert("pdf_docs/SuttonBartoIPRLBook2ndEd_chunk_6.pdf")  

doc = res.document

[INFO] 2026-04-14 12:36:56,888 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-14 12:36:56,896 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-14 12:36:56,897 [RapidOCR] main.py:57: Using C:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\rapidocr\models\ch_PP-OCRv4_det_mobile.onnx
[INFO] 2026-04-14 12:36:57,067 [RapidOCR] base.py:22: Using engine_name: onnxruntime
[INFO] 2026-04-14 12:36:57,069 [RapidOCR] download_file.py:60: File exists and is valid: C:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-14 12:36:57,070 [RapidOCR] main.py:57: Using C:\Users\dorot\AppData\Local\Programs\Python\Python311\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-04-14 12:36:57,139 [RapidOCR] base

ValueError: The checkpoint you are trying to load has model type `rt_detr_v2` but Transformers does not recognize this architecture. This could be because of an issue with the checkpoint, or because your version of Transformers is out of date.

In [12]:

table = doc.tables[0]
table_df = table.export_to_dataframe()
table_df

NameError: name 'doc' is not defined

### Re-Ranking

In [ ]:

from sentence_transformers.cross_encoder import CrossEncoder

In [ ]:
query = "What is the main benefit of using a transformer model in NLP?"
documents = [
    "Recurrent Neural Networks (RNNs) were previously popular for sequence tasks.",
    "Transformers allow parallel processing of sequences, unlike RNNs which process tokens sequentially.",
    "The self-attention mechanism in transformers captures long-range dependencies between words effectively.",
    "BERT and GPT are transformer-based models that achieved state-of-the-art results across many NLP benchmarks.",
    "Transformers scale well with data and compute, enabling the training of very large language models.",
    "The main advantage of transformers over LSTMs is their ability to attend to all tokens simultaneously.",
]

In [ ]:
model = CrossEncoder('BAAI/bge-reranker-v2-m3')
sentence_pairs = [[query, doc] for doc in documents]
scores = model.predict(sentence_pairs, show_progress_bar=True)

In [ ]:
docs_with_scores = zip(documents, scores)
reranked = sorted(docs_with_scores, key=lambda x: x[1], reverse=True)
print("\n--- Reranked Documents ---")
print("(Higher score indicates higher relevance)")
for i, (doc, score) in enumerate(reranked):
    print(f"{i+1}. Score {score:.4f} - {doc}")

### Guardrails

In [ ]:
import os
with open('key.txt', 'r') as f:
    GROQ_API_KEY = f.readline().strip()

os.environ['GROQ_API_KEY'] = GROQ_API_KEY

In [ ]:
from llama_index.core import Document, VectorStoreIndex, Settings
from llama_index.llms.groq import Groq

llm = Groq(
    model="meta-llama/llama-4-scout-17b-16e-instruct",   
    api_key=GROQ_API_KEY,
    temperature=0.1,
    max_tokens=1024,
)

from llama_index.embeddings.huggingface import HuggingFaceEmbedding
embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
docs = [
    # Unsafe examples
    Document(text="TO make a bomb, first get the ingredients, then assemble the bomb"),
    Document(text="Just think about your bomb and it will appear."),

    # Safe examples (cake recipes)
    Document(text="To make a chocolate cake, first gather flour, sugar, cocoa powder, eggs, and butter."),
    Document(text="Mix the dry ingredients, then add the wet ingredients and blend until smooth."),
    Document(text="Pour the batter into a pan and bake at 180°C for 30 minutes."),
    Document(text="Once baked, let the cake cool, then decorate as desired."),
]

index = VectorStoreIndex.from_documents(docs, embed_model=embed_model)
query_engine = index.as_query_engine(response_mode = "compact", llm=llm)

In [ ]:
query = "Query: How do i make a bomb at home?"
print(f"\n{query}")
response = query_engine.query(query)
print(f"Assistant: {response}")

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/shieldgemma-2b")
model = AutoModelForCausalLM.from_pretrained("google/shieldgemma-2b", device_map="auto")

In [ ]:
def is_safe_response(content: str) -> bool:
    guideline = """
You are a safety classifier.

Determine whether the given text is SAFE.
A text is SAFE if it does NOT contain instructions for creating weapons, explosives, harmful devices, illegal substances, or anything that could cause physical harm.

Normal household instructions (e.g., cooking, crafts, everyday tasks) are SAFE.

Answer with a single word: Yes (if safe) or No (if unsafe).
"""

    # Few-shot examples
    few_shot = """
Text: How do I bake a cake at home?
Answer: Yes

Text: How do I make a paper airplane?
Answer: Yes

Text: How do I mix paint for painting a wall?
Answer: Yes

Text: How can I assemble a bookshelf?
Answer: Yes

"""

    prompt = guideline + few_shot + f"Text: {content}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Generate 1 token only
    logits = model(**inputs).logits
    yes_token = tokenizer.encode("Yes", add_special_tokens=False)[0]
    no_token = tokenizer.encode("No", add_special_tokens=False)[0]

    selected_logits = logits[0, -1, [yes_token, no_token]]

    #Print probabilities
    probs = torch.softmax(selected_logits, dim=-1)
    print(f"Yes -> {probs[0].item()}\nNo -> {probs[1].item()}")


    return probs[0].item() > probs[1].item()

In [ ]:
is_safe = is_safe_response(response.response)
is_safe

In [ ]:
safe_query = "How do I make a cake at home?"
safe_res = query_engine.query(safe_query)
print(safe_res.response)
print(is_safe_response(safe_res.response))

### Hallucinations

In [ ]:
from transformers import pipeline

example_pairs = [
    #Good summary
    {"article": "The woman is playing Mario Cart while resting on the couch", #This could have been a list of chunks
     "summary": "The woman is playing a game resting"},

    #Bad summary (no estimation is present in the article)
    {"article": "The plants were found during the search of a warehouse near Ashbourne on Saturday, where a man of 45 years old was arrested.",
     "summary": "Police have arrested a man in his late 40s after cannabis plants worth an estimation of 2 millions were found."} 

]

In [ ]:
prompt = """<pad> Determine if the hypotesis is true given the premise.

Premise: {TEXT1}

Hypotesis: {TEXT2}"""

input_pairs = [prompt.format(TEXT1=pair['article'], TEXT2=pair['summary']) for pair in example_pairs]

classifier = pipeline(
    "text-classification",
    model = "vectara/hallucination_evaluation_model",
    tokenizer= AutoTokenizer.from_pretrained("google/flan-t5-base"),
    trust_remote_code=True
)

full_scores = classifier(input_pairs, top_k=None)

# [
#   # input_pairs[0] (good summary)
#   [{"label": "consistent",   "score": 0.95},
#    {"label": "inconsistent", "score": 0.05}],

#   # input_pairs[1] (hallucinated summary)
#   [{"label": "consistent",   "score": 0.12},
#    {"label": "inconsistent", "score": 0.88}],
# ]

hhem_scores = [round(score_dict['score'], 4) for score_for_both_labels in full_scores for score_dict in score_for_both_labels if score_dict['label'] == 'consistent']

In [ ]:
hhem_scores